# Multi-Scale Optimization Demo Notebook
This notebook demonstrates:

1) Loading the generated report and building the trade-off table.

2) Reproducing key charts (Accuracy vs Size, Accuracy vs Latency).

3) Running quick inference on the selected Cloud (Keras) and Edge/Tiny (TFLite) models.



> **Note**: Power values are **budget assumptions** for the targets (not measured).

In [ ]:

import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

REPORT_PATH = "multi_scale_optimization_report.json"

# Force CPU for latency stability (safe on Mac M1/CPU-only envs)
def _pin_cpu_only():
    try:
        tf.config.set_visible_devices([], "GPU")
        tf.config.set_visible_devices([], "TPU")
    except Exception:
        pass

_pin_cpu_only()

# Data utils
def load_cifar10_test(num_samples=None):
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    x_test = x_test.astype("float32") / 255.0
    y_test = y_test.reshape(-1)
    if num_samples is not None:
        x_test, y_test = x_test[:num_samples], y_test[:num_samples]
    return x_test, y_test

def evaluate_keras_latency_accuracy(model, x, y, warmup=10, runs=50, batch_size=1):
    y_prob = model.predict(x, batch_size=batch_size, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    acc = float(np.mean(y_pred == y))

    sample = x[:batch_size]
    for _ in range(warmup):
        _ = model.predict(sample, batch_size=batch_size, verbose=0)
    t0 = time.perf_counter()
    for _ in range(runs):
        _ = model.predict(sample, batch_size=batch_size, verbose=0)
    t1 = time.perf_counter()
    latency_ms = (t1 - t0) / runs * 1000.0
    return acc, latency_ms

def _quantize_input_like(interpreter, x_batch):
    d = interpreter.get_input_details()[0]
    scale, zero = d["quantization"]
    dtype = d["dtype"]
    if scale == 0:
        return x_batch.astype(dtype)
    q = np.round(x_batch / scale + zero).astype(dtype)
    if np.issubdtype(dtype, np.integer):
        info = np.iinfo(dtype)
        q = np.clip(q, info.min, info.max)
    return q

def evaluate_tflite_latency_accuracy(tflite_path, x, y, warmup=20, runs=50, batch_size=1):
    interpreter = tf.lite.Interpreter(model_path=tflite_path, num_threads=1)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    xb = x[:batch_size]
    inp = _quantize_input_like(interpreter, xb)
    for _ in range(warmup):
        interpreter.set_tensor(input_details[0]["index"], inp)
        interpreter.invoke()
    t0 = time.perf_counter()
    for _ in range(runs):
        interpreter.set_tensor(input_details[0]["index"], inp)
        interpreter.invoke()
    t1 = time.perf_counter()
    lat_ms = (t1 - t0) / runs * 1000.0

    N = min(2000, x.shape[0])
    correct = 0
    for i in range(N):
        xi = _quantize_input_like(interpreter, x[i:i+1])
        interpreter.set_tensor(input_details[0]["index"], xi)
        interpreter.invoke()
        logits = interpreter.get_tensor(output_details[0]["index"])
        if int(np.argmax(logits, axis=1)[0]) == int(y[i]):
            correct += 1
    acc = correct / N
    return float(acc), float(lat_ms)

# Load report and build table
with open(REPORT_PATH, "r") as f:
    report = json.load(f)

res = report["optimization_results"]
rows = []
power_map_mw = {"cloud_server": 50000.0, "edge_device": 2000.0, "microcontroller": 10.0}
for t, d in res.items():
    rows.append({
        "Target": t,
        "Strategy": d["optimization_strategy"],
        "Model Path": d["model_path"],
        "Model Size (MB)": d["model_size_mb"],
        "Accuracy": d["accuracy"],
        "Latency (ms)": d["estimated_latency_ms"],
        "Memory (MB)": d["memory_usage_mb"],
        "Power Estimate": f'{power_map_mw[t]/1000.0:.1f} W' if t=="cloud_server" else f'{power_map_mw[t]:.0f} mW',
    })
df = pd.DataFrame(rows)
df


In [ ]:

# Charts
plt.figure()
plt.scatter(df["Model Size (MB)"], df["Accuracy"])
for i, row in df.iterrows():
    plt.text(row["Model Size (MB)"], row["Accuracy"], row["Target"])
plt.xlabel("Model Size (MB)"); plt.ylabel("Accuracy"); plt.title("Accuracy vs. Model Size")
plt.show()

plt.figure()
plt.scatter(df["Latency (ms)"], df["Accuracy"])
for i, row in df.iterrows():
    plt.text(row["Latency (ms)"], row["Accuracy"], row["Target"])
plt.xlabel("Latency (ms)"); plt.ylabel("Accuracy"); plt.title("Accuracy vs. Latency")
plt.show()


## Quick Inference Checks
Run a few predictions on the selected models to verify accuracy and approximate latency on this machine.

In [ ]:

x_test, y_test = load_cifar10_test(5000)

# Keras (Cloud)
try:
    cloud_path = res["cloud_server"]["model_path"]
    cloud_model = tf.keras.models.load_model(cloud_path)
    acc, lat = evaluate_keras_latency_accuracy(cloud_model, x_test, y_test)
    print("Cloud:", cloud_path, "acc=", round(acc,4), "lat(ms)=", round(lat,4))
except Exception as e:
    print("Cloud model not available:", e)

# TFLite (Edge)
try:
    edge_path = res["edge_device"]["model_path"]
    a, l = evaluate_tflite_latency_accuracy(edge_path, x_test, y_test)
    print("Edge:", edge_path, "acc=", round(a,4), "lat(ms)=", round(l,4))
except Exception as e:
    print("Edge model not available:", e)

# TFLite (Tiny)
try:
    tiny_path = res["microcontroller"]["model_path"]
    a, l = evaluate_tflite_latency_accuracy(tiny_path, x_test, y_test)
    print("Tiny:", tiny_path, "acc=", round(a,4), "lat(ms)=", round(l,4))
except Exception as e:
    print("Tiny model not available:", e)



## Scenario Recommendations (A/B/C)
- **A Real-time Video (<50ms):** Prefer **Edge** for on-device inference; keep **Cloud** for backup/retraining.

- **B IoT Sensor (<1mW):** Prefer **Tiny INT8** with occasional cloud sync.

- **C Mobile (offline):** Use **multi-tier**: local FP16/INT8 + offline cache + cloud sync when online.
